# Exploración de datos OBD-II y análisis temporal

**Notebook** `01_exploracion_obd` · **versión** 1.1 · **fecha** 14 de noviembre de 2026

| | |
|---|---|
| **Modalidad** | REANÁLISIS |
| **Experimentos** | E2 (generación), E4 (exploratorio), E5 (ventanado y reparto) |
| **Conjunto** | `data/samples/synthetic_sample.parquet`, 18 sesiones · sesiones de calentamiento generadas en el notebook |
| **Protocolo** | ventana 120 s, paso 60 s, reparto por sesión |
| **Estado** | E2, E4 y E5 cerradas y validadas |
| **Pregunta de investigación** | PI.1: que el diagnóstico a partir de OBD-II supere a la clase mayoritaria y que su confianza se pueda calibrar |
| **Evidencia original** | `docs/experiments/E4_eda_temporal.md`, `E5_preprocesamiento.md`, `artifacts/tables/E4_*.csv` |

---

## En un minuto

- **Pregunta.** Si las señales OBD-II tienen estructura suficiente para distinguir una condición
  de otra y qué forma tiene.
- **Qué se hace.** Se recalculan descriptivos sobre una muestra de 18 sesiones y se leen las cifras
  publicadas de E4 y E5.
- **Resultado.** Las condiciones desplazan las señales declaradas sin que ninguna las separe sola
  con un umbral, con **cinco firmas distintas** en su señal principal (tres formas temporales medidas
  como desviación, notebook 06). E5 eligió ventanas de 120 s: desde 60 s las cinco superan el suelo
  de ruido y a 120 s el margen del fallo de encendido es mayor, a cambio de menos ventanas.
- **Lectura.** Al menos dos condiciones tienen ventanas etiquetadas **sin señal observable del
  fallo**. Etiqueta y señal no van uno a uno.
- **Límite.** Datos sintéticos de un generador no calibrado contra datos reales.

## Cómo leer este notebook

**Modalidad:** reanálisis (apartado 4); el apartado 8 es una demostración ilustrativa y el 10,
visualización de evidencia histórica. Los descriptivos recalculados **no reproducen** E4 y no deben
citarse como las cifras publicadas. **Ejecución:** CPU, sin red, menos de dos minutos.

- **1–4.** Introducción, hipótesis, fundamento y entorno.
- **5–9.** Los conjuntos sintéticos; una sesión sana; la firma de cada fallo; el termostato;
  ninguna señal separa sola.
- **10–12.** Separabilidad publicada de E4; las dos mezclas pobres; correlaciones.
- **13–15.** La ventana de 120 s; de la señal al descriptor; el reparto por sesión.
- **16–20.** Limitaciones, discusión, conclusiones, reproducibilidad y referencias.

## 1. Introducción: por qué hace falta mirar antes de modelar

Es la pregunta previa a cualquier modelo: si las señales de un motor con fuga de vacío fueran
indistinguibles de las de uno sano, no habría nada que aprender y el resto del trabajo carecería de
objeto. La respuesta no es un sí o un no. Hay condiciones que se ven a simple vista y condiciones
cuya firma es una **ausencia**: algo que debería ocurrir y no ocurre. Esas son las interesantes.

Un modelo entrenado sobre datos que nadie ha mirado puede obtener métricas altas por motivos
que no tienen nada que ver con el problema. Si el generador dejara, por ejemplo, una señal que tomara
un valor imposible solo durante un fallo, cualquier clasificador la encontraría y el resultado
mediría la calidad del generador, no la del diagnóstico.

Lo que no sabemos antes de empezar:

1. **Si las relaciones entre señales están presentes.** Un generador de variables independientes al
   azar no tendría estructura que aprender y la atribución no tendría nada que señalar.
2. **Si alguna condición se separa trivialmente.** Sería un artefacto del generador.
3. **Cuánto dura la firma de cada fallo**, que es lo que determina el tamaño de ventana.
4. **Si todas las condiciones se manifiestan igual en todos los regímenes.** No lo hacen: esa es
   la observación más consecuente de este notebook.

## 2. Objetivo e hipótesis

**Objetivo.** Caracterizar las señales, sus relaciones y la forma temporal de cada condición.
Fijar con evidencia el tamaño de ventana.

**Hipótesis evaluada.** Las condiciones de fallo producen desplazamientos observables en las señales
que su ficha declara afectadas y ese desplazamiento es suficiente para separarlas de la condición
normal sin que ninguna señal lo haga por sí sola con un umbral.

**Variables observadas.** Las doce señales del catálogo, la condición, el régimen y el tiempo.

**Criterios de evaluación.** Un desplazamiento se considera observable si el tamaño del efecto
(delta de Cliff) lo separa de la condición normal. La separación se considera **trivial** (y por
tanto un defecto) si existe un umbral de una sola señal que clasifique perfectamente una condición.

**Qué cuestionaría la hipótesis.** Que ninguna señal se desplace en una condición, o que alguna la
separe perfectamente. Las dos cosas se comprueban en este notebook.

## 3. Fundamento: lo necesario para leer lo que sigue

**OBD-II.** Norma de diagnóstico a bordo obligatoria en los vehículos de pasajeros desde finales de
los noventa. Expone un conjunto de parámetros (*PID*) que la centralita publica a petición: régimen,
velocidad, carga calculada, temperatura de refrigerante, caudal de aire, ajustes de combustible,
tensión de la sonda lambda y avance de encendido, entre otros. Aquí se usan doce, con su
correspondencia a los PID canónicos declarada en `obd/signals.py`.

**Serie temporal y ventana.** Las señales llegan como series muestreadas a 1 Hz. Un clasificador de
ventanas no mira la muestra suelta, sino un **tramo** de duración fija. La elección de esa duración
no es libre: demasiado corta y la firma no cabe; demasiado larga y hay pocas ventanas por sesión.

**Descriptores de ventana.** Resumir un tramo de 120 × 12 valores en un vector de números: media,
desviación, pendiente, autocorrelación, número de cruces. Es una **representación** del tramo:
cambiarla cambia lo que un modelo puede ver.

**Delta de Cliff.** Medida no paramétrica del tamaño de un efecto: la probabilidad de que un valor
tomado de un grupo supere a uno del otro, menos la probabilidad contraria. Va de −1 a 1. *Por qué
esta y no la diferencia de medias:* no supone normalidad ni igualdad de varianzas; aquí las
distribuciones no son normales.

**Fuga temporal.** Repartir ventanas de una misma sesión entre entrenamiento y evaluación haría que
el modelo viera en el examen tramos solapados con los que estudió: la evaluación mediría memoria, no
capacidad de generalizar.

## 4. Preparación del entorno

In [ ]:
# Se busca hacia arriba la carpeta del paquete (la que contiene `src/mecanico_ia_exp`),
# así el notebook funciona se abra Jupyter desde donde se abra, sin rutas absolutas.
import sys
from pathlib import Path

RAIZ = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "mecanico_ia_exp").is_dir()),
    Path.cwd(),
)
if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))

try:
    from mecanico_ia_exp.notebooks import soporte
except ModuleNotFoundError as error:
    raise SystemExit(
        f"No se encuentra el paquete del laboratorio desde {Path.cwd()}.\n"
        "Abra Jupyter dentro del repositorio clonado, o instálelo con uv sync --extra nb."
    ) from error

soporte.imprimir_entorno()

In [ ]:
soporte.declarar_modalidad(
    "REANÁLISIS",
    "los descriptivos se recalculan aquí sobre la muestra versionada de 18 sesiones; "
    "NO son las cifras publicadas de E4, que se midieron sobre synthetic_v1 con 36 sesiones. "
    "Las figuras publicadas de E4 están en artifacts/figures/ y se muestran más adelante "
    "sin recalcularlas",
)

## 5. El conjunto: procedencia y estructura

La muestra que viaja con el repositorio son 18 sesiones de 600 s, tres por condición. Sale del mismo
generador y de la misma semilla global que el conjunto oficial.

**Lo que hay que distinguir (y el trabajo tardó en distinguirlo):** hay tres conjuntos sintéticos y
**no salen del mismo generador**.

| Conjunto | Sesiones | Generado | Estado del generador |
|---|---|---|---|
| `synthetic_v1` | 36 | 01-09-2026 | **pre-calibración**: no se reconstruye con el generador publicado |
| `synthetic_v2` | 72 | 14-09-2026 | post-calibración · **conjunto oficial** |
| `synthetic_v3` | 216 | **01-09-2026** | **pre-calibración**: ralentí en 820 rpm, sin respaldo documental |

La constante `GENERATOR_VERSION` valió `"1.0.0"` en los tres y no los distingue (notebook 02,
apartado 13). Comparar `v2` con `v3` cambia el volumen **y** el fenómeno; esa confusión invalidó una
conclusión publicada. Aquí no se mezclan.

In [ ]:
datos = soporte.muestra_redistribuible()

print(f"{len(datos):,} muestras · {datos.session_id.nunique()} sesiones · "
      f"{datos.fault_id.nunique()} condiciones")
print(f"valores ausentes: {int(datos.isna().sum().sum())}")
print()

from mecanico_ia_exp.obd.signals import SIGNAL_NAMES, SIGNALS_BY_NAME

rangos = datos[list(SIGNAL_NAMES)].agg(["min", "max", "mean", "std"]).T
rangos["rango físico"] = [
    f"{SIGNALS_BY_NAME[s].minimum:g} … {SIGNALS_BY_NAME[s].maximum:g}" for s in rangos.index
]
rangos["unidad"] = [SIGNALS_BY_NAME[s].unit for s in rangos.index]
rangos.round(2)

**Qué muestra.** Ninguna señal sale de su rango físico y no hay valores ausentes. El recorte a
rango no es decorativo: la inyección de un fallo puede empujar una señal fuera de lo posible (un
ajuste de combustible del 140 %) y dejarla pasar regalaría al modelo una pista que en datos reales
nunca existiría, porque un sensor real satura.

A continuación, el reparto por condición y régimen, donde se ve el equilibrio buscado y una
ausencia.

In [ ]:
import pandas as pd

reparto = (
    datos.drop_duplicates("session_id")
    .pivot_table(index="fault_id", columns="regime", values="session_id", aggfunc="count")
    .fillna(0)
    .astype(int)
)
reparto["total"] = reparto.sum(axis=1)
reparto

**Qué muestra.** Tres sesiones por condición, repartidas entre **tres** regímenes: urbano,
autopista y ralentí. **No hay sesiones de calentamiento.**

**Qué podemos interpretar.** Es aritmética, no un sesgo del muestreo: el plan equilibrado recorre
cuatro regímenes en ciclo y con tres sesiones por condición solo alcanza a los tres primeros. El
conjunto oficial, con doce por condición, sí los cubre los cuatro.

**Qué no podemos afirmar.** Nada sobre el comportamiento en calentamiento a partir de esta muestra.
El calentamiento es el régimen donde una de las condiciones se comporta de forma distinta;
el apartado 8 lo ilustra con sesiones **generadas** en el notebook.

## 6. Exploración: las señales en el tiempo

**Qué queremos investigar.** Si la estructura de las señales es plausible y si las relaciones que el
generador declara construir (acelerador↔régimen, régimen↔caudal, carga↔presión) se ven.

**Qué datos utilizamos.** Una sesión sana del régimen urbano, completa.

**Qué variables se representan.** Seis de las doce señales, en el tiempo y con el eje compartido.
Seis y no doce para que se lean.

In [ ]:
import matplotlib.pyplot as plt

from mecanico_ia_exp.visualization.style import CATEGORICAL, apply_style

apply_style()

SENALES_VISTA = [
    ("vehicle_speed", "Velocidad", "km/h"),
    ("rpm", "Régimen del motor", "rpm"),
    ("engine_load", "Carga calculada", "%"),
    ("maf", "Caudal másico de aire", "g/s"),
    ("coolant_temperature", "Temperatura de refrigerante", "°C"),
    ("stft_bank1", "Ajuste de combustible a corto plazo", "%"),
]

sana = datos[datos.session_id == "F00_normal__urban__00"].sort_values("timestamp")

fig, ejes = plt.subplots(len(SENALES_VISTA), 1, figsize=(9.5, 8.4), sharex=True)
for eje, (senal, titulo, unidad), color in zip(ejes, SENALES_VISTA, CATEGORICAL, strict=False):
    eje.plot(sana.timestamp, sana[senal], color=color, linewidth=0.9)
    eje.set_ylabel(f"{titulo}\n({unidad})", fontsize=7)
ejes[-1].set_xlabel("tiempo desde el arranque (s)")
fig.suptitle(
    "Sesión sin fallo, régimen urbano: seis de las doce señales OBD-II",
    fontsize=10.5, fontweight="bold", x=0.0, ha="left",
)
fig.supxlabel(
    "E2 generación · synthetic_sample · sesión F00_normal__urban__00 · 600 muestras a 1 Hz",
    fontsize=6.5, x=0.0, ha="left",
)
fig.tight_layout()
soporte.exportar(fig, "FIG-NB01-01_sesion_sana", subdirectorio="01_exploracion_obd")
plt.show()

**Qué muestra.** La velocidad alterna tramos de marcha y paradas, propio del régimen urbano. El
régimen del motor la sigue con saltos en los cambios de relación y cae a un ralentí plano cuando el
vehículo se detiene. La carga y el caudal acompañan al régimen. La temperatura de refrigerante parte
alta (en este régimen el motor arranca caliente) y se mantiene. El ajuste de combustible a corto
plazo oscila en torno a cero sin desplazamiento.

**Qué podemos interpretar.** Las relaciones entre señales que el generador dice construir están
presentes y son del signo esperado.

**Qué no podemos afirmar.** Que estas trayectorias se parezcan a las de un vehículo real: el
generador no está calibrado contra datos reales (apartado 16). Tampoco se puede afirmar nada sobre
otros regímenes a partir de una sesión.

## 7. Normal frente a fallo: dónde se ve la firma

**Qué queremos investigar.** Si la firma declarada de cada fallo aparece donde su ficha dice y qué
forma tiene en el tiempo: no todas las firmas son un escalón.

**Qué datos utilizamos.** Para cada condición, una sesión del mismo régimen que la sana, de modo que
la única diferencia sea la condición.

**Qué variables se representan.** La señal que la ficha del fallo declara como principal, con la
trayectoria sana y la averiada en el mismo eje para que la escala no engañe.

In [ ]:
from mecanico_ia_exp.obd.faults import FAULTS

# Para cada fallo, su señal principal declarada y una sesión del mismo régimen.
PRINCIPAL = {
    "F01_vacuum_leak": ("stft_bank1", "Ajuste a corto plazo", "%"),
    "F02_maf_drift": ("ltft_bank1", "Ajuste a largo plazo", "%"),
    "F03_thermostat_stuck": ("coolant_temperature", "Temperatura de refrigerante", "°C"),
    "F04_o2_lag": ("oxygen_sensor", "Tensión de la sonda lambda", "V"),
    "F05_misfire": ("rpm", "Régimen del motor", "rpm"),
}

fig, ejes = plt.subplots(len(PRINCIPAL), 1, figsize=(9.5, 8.8), sharex=True)
for eje, (fid, (senal, titulo, unidad)) in zip(ejes, PRINCIPAL.items(), strict=False):
    con_fallo = datos[datos.session_id == f"{fid}__urban__00"].sort_values("timestamp")
    eje.plot(sana.timestamp, sana[senal], color=CATEGORICAL[0],
             linewidth=0.9, label="sin fallo", alpha=0.85)
    eje.plot(con_fallo.timestamp, con_fallo[senal], color=CATEGORICAL[1],
             linewidth=0.9, label=FAULTS[fid].fault_name)
    inicio = float(con_fallo.loc[con_fallo.fault_active, "timestamp"].min())
    eje.axvline(inicio, color="#898781", linestyle="--", linewidth=0.8)
    eje.set_ylabel(f"{titulo}\n({unidad})", fontsize=7)
    # La leyenda va encima del panel y no dentro: dentro tapaba parte de las curvas.
    eje.legend(loc="lower left", bbox_to_anchor=(0.0, 0.98), fontsize=6.5, ncols=2,
               frameon=False, borderaxespad=0.0)
ejes[-1].set_xlabel("tiempo desde el arranque (s)")
fig.suptitle(
    "La firma de cada condición en la señal que su ficha declara principal",
    fontsize=10.5, fontweight="bold", x=0.0, ha="left",
)
fig.supxlabel(
    "E2 generación · synthetic_sample · régimen urbano · "
    "línea discontinua: instante de entrada del fallo (120 s)",
    fontsize=6.5, x=0.0, ha="left",
)
fig.tight_layout()
soporte.exportar(fig, "FIG-NB01-02_normal_vs_fallo", subdirectorio="01_exploracion_obd")
plt.show()

**Qué muestra.** Cinco firmas distintas en la señal principal:

- **Fuga de vacío**: el ajuste a corto plazo salta a positivo y se queda. Un escalón.
- **Deriva del caudalímetro**: el ajuste a largo plazo sube en **rampa**, no de golpe. El integrador
  aprende durante minutos.
- **Termostato bloqueado**: la temperatura queda **recortada** a un techo.
- **Sonda lenta**: la oscilación característica se **amortigua**; el valor medio no se mueve.
- **Fallo de encendido**: caídas **breves y aisladas** del régimen. La mayor parte del tiempo la
  señal es indistinguible de la sana.

**Qué podemos interpretar.** Cada firma se ve distinta en su señal. Medidas como desviación frente
al contrafactual se agrupan en tres formas temporales: progresiva, sostenida desde la entrada y
episódica (notebook 06, apartado 7.1). Eso tiene una consecuencia práctica: un descriptor de **nivel medio** captura bien la fuga de vacío y es
casi ciego al fallo de encendido, cuyos episodios se diluyen en la media de la ventana. La
representación que se elija no es neutral respecto a qué fallos se pueden detectar.

**Qué no podemos afirmar.** Que estas cinco sesiones sean representativas de su condición: es una
sesión por fallo. Tampoco que la firma sea igual en otros regímenes. El termostato, en particular,
no lo es, como se ve a continuación. Los episodios del fallo de encendido caen donde los pone
esta semilla; otra los pondría en otros. La versión publicada de esta comparación, sobre `synthetic_v1`, se muestra en el
apartado 10.

## 8. Una firma que es una ausencia: el termostato

El fallo consiste en imponer un **techo** de 71 °C a la temperatura del refrigerante. Mientras el
motor no habría superado ese techo por sí mismo, el fallo **no impone nada**: la señal de un motor
averiado es idéntica, valor a valor, a la de uno sano.

En los regímenes urbano, autopista y ralentí el motor arranca ya caliente, por encima del techo, de
modo que el recorte actúa desde el primer instante. En **calentamiento** arranca a temperatura
ambiente y tarda minutos en llegar.

Como la muestra versionada no tiene sesiones de calentamiento, se **regeneran** aquí. **No es un
escenario nuevo ni un experimento**. Es el mismo
generador, con sus parámetros por omisión y una semilla del laboratorio, reproduciendo el fenómeno
que ya está medido y documentado en `docs/methodology/expresion-efectiva-del-fallo.md`. No se
persiste ningún fichero, no se crea ningún conjunto y no se mide ninguna cifra nueva: se dibuja una
que ya existe. Modalidad de esta sección: **demostración ilustrativa**.

In [ ]:
from mecanico_ia_exp.obd.synthetic import SessionSpec, generate_session

# Dos sesiones de calentamiento con la MISMA semilla: la única diferencia es la
# condición. Eso es lo que permite comparar trayectorias sin más confusores.
SEMILLA_ILUSTRATIVA = 20260901

sana_w = generate_session(SessionSpec(
    session_id="ilustrativa_sana", regime="warmup", duration_s=600.0,
    seed=SEMILLA_ILUSTRATIVA, fault_id="F00_normal",
))
fallo_w = generate_session(SessionSpec(
    session_id="ilustrativa_termostato", regime="warmup", duration_s=600.0,
    seed=SEMILLA_ILUSTRATIVA, fault_id="F03_thermostat_stuck", fault_onset_s=120.0,
))

TECHO = FAULTS["F03_thermostat_stuck"].default_params["temp_cap_c"]

fig, (arriba, abajo) = plt.subplots(2, 1, figsize=(9.5, 5.4), sharex=True,
                                    height_ratios=[2.1, 1])
# La sana va debajo y más gruesa: durante los primeros minutos las dos
# trayectorias son la MISMA; si se dibujaran igual de finas la de arriba
# taparía a la de abajo y parecería que solo hay una línea.
arriba.plot(sana_w.timestamp, sana_w.coolant_temperature, color=CATEGORICAL[0],
            label="sin fallo", linewidth=2.6, alpha=0.55)
arriba.plot(fallo_w.timestamp, fallo_w.coolant_temperature, color=CATEGORICAL[1],
            label="termostato bloqueado", linewidth=1.0)
arriba.axhline(TECHO, color="#898781", linestyle=":", linewidth=0.9)
arriba.text(8, TECHO + 1.5, f"techo del fallo: {TECHO:g} °C", fontsize=6.8, color="#52514e")
arriba.axvline(120, color="#898781", linestyle="--", linewidth=0.8)
arriba.text(126, 30, "entrada del fallo", fontsize=6.8, color="#52514e")
arriba.set_ylabel("Temperatura de\nrefrigerante (°C)", fontsize=7.5)
arriba.legend(loc="lower right", fontsize=7)

# Abajo: la desviación frente al contrafactual, que es cero mientras el fallo no
# impone nada. No es una estimación: es la diferencia con la trayectoria que el
# mismo motor habría seguido con la misma semilla.
abajo.fill_between(fallo_w.timestamp, 0, fallo_w.fault_deviation_max,
                   color=CATEGORICAL[1], alpha=0.35)
abajo.axhline(1.0, color="#898781", linestyle=":", linewidth=0.9)
abajo.text(8, 1.6, "umbral de observabilidad", fontsize=6.5, color="#52514e")
abajo.set_ylabel("Alteración frente al\ncontrafactual (umbrales)", fontsize=7.5)
abajo.set_xlabel("tiempo desde el arranque (s)")
abajo.set_yscale("symlog", linthresh=1.0)
abajo.set_ylim(bottom=0)

fig.suptitle(
    "El termostato bloqueado no altera nada hasta que el motor alcanza su techo",
    fontsize=10.5, fontweight="bold", x=0.0, ha="left",
)
fig.supxlabel(
    "E2 generación · DEMOSTRACIÓN ILUSTRATIVA: sesiones generadas en este notebook, "
    f"semilla {SEMILLA_ILUSTRATIVA}, régimen warmup · no son cifras publicadas",
    fontsize=6.5, x=0.0, ha="left",
)
fig.tight_layout()
soporte.exportar(fig, "FIG-NB01-03_termostato_calentamiento",
                 subdirectorio="01_exploracion_obd")
plt.show()

activas = fallo_w[fallo_w.fault_active]
print(f"muestras con el fallo aplicado:   {len(activas)}")
print(f"muestras que alteran la señal:    {int(activas.fault_expressed.sum())} "
      f"({activas.fault_expressed.mean():.0%})")

**Qué muestra.** Arriba, las dos trayectorias son **la misma línea** durante unos trescientos
segundos tras la entrada del fallo y se separan solo cuando la sana alcanza el techo. Abajo, la
alteración frente al contrafactual es **exactamente cero** en ese tramo.

**Qué podemos interpretar.** La firma de este fallo es una ausencia y la ausencia solo es
observable cuando hay algo que ausentarse. Durante el calentamiento la etiqueta dice “termostato
bloqueado” y la señal no contiene ninguna información que lo distinga de un motor sano. Ningún
clasificador puede separar dos señales idénticas. Es una propiedad del mecanismo tal como el
generador lo implementa (notebook 06, apartado 8), no de los termostatos reales.

**Qué no podemos afirmar.** Que esto explique por sí solo los errores que los modelos cometen en esta
condición: lo que establece es que **existe** un tramo sin señal, no cuántos de los errores caen en
él. La identificación una a una exige cruzar las predicciones con la expresión ventana a ventana.
Ese cruce está diseñado y **no ejecutado** (`docs/diseno-pi1/08-analisis-por-fallo.md`).

Tampoco debe concluirse de aquí que el contexto temporal sea valioso porque esta condición lo
requiera; por qué la ventaja del modelo recurrente en esta clase no lo demuestra se explica en el
notebook 02, apartado 8.

## 9. ¿Separa alguna señal por sí sola?

**Qué queremos investigar.** Si existe una señal delatora: un umbral único que clasifique
perfectamente alguna condición. Sería un artefacto del generador y haría trivial el problema.

**Qué datos utilizamos.** Las doce señales sobre las 18 sesiones, muestra a muestra. Un umbral
que separe perfectamente existe si y solo si los rangos de las dos clases son **disjuntos**, de modo
que basta comparar dos extremos por señal y condición y la respuesta es un sí o un no.

In [ ]:
from mecanico_ia_exp.obd.auditoria import auditar

# La auditoría del generador, la misma que se ejecuta antes de entrenar.
# Doce comprobaciones; aquí se muestran todas, también las que pasan.
for comprobacion in auditar(datos):
    print(comprobacion)

**Qué muestra.** Las doce comprobaciones con su estado: seis `PASS`, cinco `NO_VERIFICABLE` y una
`INFORMATIVA`. Tres de las cinco no verificables (5, 11 y 12) lo son porque esta muestra no trae
manifiesto de procedencia (se generó antes de que existiera); la 6, porque no tiene sesiones sanas
de calentamiento; la 8, porque no trae las columnas de expresión del fallo.

**Qué podemos interpretar.** Ninguna de las doce señales separa por sí sola ninguna de las cinco
condiciones con un solo umbral **muestra a muestra**: el problema no es trivial en ese sentido.
Agregadas por bloque es otra cosa: en E4, la media de `stft_bank1` por bloque separa por completo la
fuga de vacío y la deriva del caudalímetro del motor sano (delta de Cliff 1,00 en las dos,
`E4_separabilidad.csv`). Las relaciones entre señales están
presentes con el signo esperado. El ralentí sin fallo cae dentro del rango que el manual documenta
para este motor, 640–740 rpm. Esa comprobación habría detectado el problema de `synthetic_v3` el
día que se generó.

**Qué no podemos afirmar.** Que no exista una **combinación** de señales que separe trivialmente.
La comprobación es de una señal y un umbral; una regla de dos señales no la detectaría. Tampoco que
el conjunto oficial pase igual: se ha comprobado sobre esta muestra.

Dos comprobaciones merecen comentario porque informan y no aprueban:

- **La expresión efectiva de los fallos** (aquí no verificable) se mide y se publica, no se
  aprueba. Una condición que se
  exprese poco no es un defecto del conjunto: es el fenómeno. Lo que sería un defecto es tratar la
  etiqueta como prueba de que el fallo se manifestó.
- **La variabilidad entre sesiones** se informa por el mismo motivo.

## 10. Separabilidad por señal y condición: las figuras publicadas de E4

El apartado 9 comprueba que ninguna señal separa sola una condición; E4 midió **cuánto** separa
cada una. Sus figuras se muestran tal como se publicaron, sin recalcular: **visualización de
evidencia histórica**. La primera es la versión publicada de la comparación del apartado 7.

In [ ]:
from IPython.display import Image, display

for nombre, pie in [
    ("FIG-E4-01_series_normal_vs_fallo",
     "E4 · synthetic_v1 · series normales frente a series con fallo"),
    ("FIG-E4-02_separabilidad",
     "E4 · synthetic_v1 · separabilidad por señal y condición, delta de Cliff"),
]:
    print(pie)
    display(Image(filename=str(soporte.figura(nombre)), width=760))

**Qué muestran.** Las series con y sin fallo publicadas por E4 y el tamaño del efecto por señal y
condición.

**Qué podemos interpretar.** Que el hallazgo central de E4 se sostiene: la deriva del caudalímetro
**no se ve en el caudalímetro**. El efecto aparece en los ajustes de combustible, porque la
centralita compensa la subestimación del aire inyectando más. La firma de un fallo no está
necesariamente en la señal del componente que falla.

**Qué no podemos afirmar.** Estas figuras salen de `synthetic_v1` (con 36 sesiones) y no de
`synthetic_v2`. No deben leerse como si fueran del conjunto oficial.

La segunda figura se redibuja porque es legible en pantalla y no en papel: se publicó a 9,6 pulgadas
de ancho y el documento tiene 5,9, de modo que al entrar se escala al 61 % y una etiqueta de 7
puntos se imprime a 4,3. Se redibuja **a la medida de la página**, con los mismos datos de E4 y sin
recalcular ningún tamaño de efecto: para cada una de las doce señales y cada una de las cinco
condiciones, el delta de Cliff frente a la condición sin fallo, calculado por separado sobre la
media del bloque y sobre su dispersión. Se etiquetan solo las celdas con efecto grande,
|delta| ≥ 0,474, que es la convención de lectura habitual.

In [ ]:
from mecanico_ia_exp.visualization.memoria import (
    fig_separabilidad_dispersion,
    fig_separabilidad_nivel,
)

# Dos mapas y no dos paneles de uno: comparten las doce filas y las cinco
# columnas; apilados solo el de arriba conservaba los nombres de las señales.
# Nombres en español de las doce señales, con su identificador entre paréntesis
# para poder cruzarlas con las tablas. Solo cambian los rótulos, no los datos.
NOMBRE_SENAL = {
    "rpm": "régimen del motor",
    "vehicle_speed": "velocidad",
    "engine_load": "carga calculada",
    "throttle_position": "posición de mariposa",
    "coolant_temperature": "temperatura del refrigerante",
    "intake_air_temperature": "temperatura del aire de admisión",
    "maf": "caudal de aire (MAF)",
    "map": "presión del colector (MAP)",
    "stft_bank1": "ajuste a corto plazo (STFT)",
    "ltft_bank1": "ajuste a largo plazo (LTFT)",
    "oxygen_sensor": "sonda lambda",
    "ignition_timing": "avance del encendido",
}
SEPARABILIDAD = [
    (fig_separabilidad_nivel, "FIG-NB01-04_separabilidad_nivel",
     "Separabilidad por el nivel: delta de Cliff de la media del bloque"),
    (fig_separabilidad_dispersion, "FIG-NB01-05_separabilidad_dispersion",
     "Separabilidad por la dispersión: delta de Cliff de la desviación típica del bloque"),
]
for funcion, nombre, titulo in SEPARABILIDAD:
    fig = funcion()
    mapa = fig.axes[0]
    mapa.set_yticklabels([NOMBRE_SENAL.get(t.get_text(), t.get_text())
                          for t in mapa.get_yticklabels()], fontsize=7.5)
    mapa.set_xlabel("condición de fallo, comparada con la condición sin fallo")
    fig.suptitle(titulo, fontsize=9.5, fontweight="bold", x=0.01, y=0.99,
                 ha="left", va="bottom")
    fig.text(
        0.01, 0.0,
        "E4 · synthetic_v1 · fuente: artifacts/tables/E4_separabilidad.csv · rojo: valores "
        "mayores que sin fallo; azul: menores · se rotulan las celdas con |delta| ≥ 0,474 "
        "(efecto grande)",
        fontsize=6.5, ha="left", va="top",
    )
    soporte.exportar(fig, nombre, subdirectorio="01_exploracion_obd")
    plt.show()

**Qué muestran.** Las cinco mejores celdas, una por condición, reparten los fallos en dos grupos.
Tres se separan por el **nivel** y con efecto máximo o casi: la fuga de vacío y la deriva del
caudalímetro en el ajuste de combustible a corto plazo; el termostato, en la temperatura del
refrigerante. La sonda lenta alcanza el efecto máximo **solo por la dispersión**, con un nivel de
+0,29 que no distingue nada. El fallo de encendido tiene un único efecto grande, +0,58, el más débil
de los cinco.

**Qué podemos interpretar.** Que los descriptores por ventana no podían limitarse a medias sin dejar
dos condiciones indetectables por construcción. Es la justificación empírica de la decisión de
incluir dispersión, extremos y conteos de eventos en el vector de descriptores.

**Qué no podemos afirmar.** El delta de Cliff ordena pero no explica: un efecto grande en una señal
no implica que el modelo la use. El análisis de atribución muestra un caso en que no la usa. La
medición es sobre bloques de 60 segundos y no sobre las ventanas de 120 con que se entrena. Sale
de `synthetic_v1`, que es **pre-calibración** y no se reconstruye con el generador publicado.

**Relación con la pregunta de investigación.** Delimitan por anticipado qué afirmaciones sobre una señal pueden
tener soporte observacional bajo PI.1 y cuáles no. En la memoria son las **Figuras 5.2 y 5.3**.

## 11. Qué distingue entre sí las dos averías de mezcla pobre

Los dos apartados anteriores preguntan si una señal separa una
condición de la condición sin fallo. Hay un caso en que esa pregunta se queda corta. La fuga de
vacío y la deriva del caudalímetro empobrecen las dos la mezcla y la centralita responde a las dos
subiendo las correcciones de combustible: las dos se separan de la condición sin fallo por lo mismo
y con el efecto máximo. Eso dice que las dos se detectan; no dice cómo se distinguen **entre sí**,
que es lo que decide la reparación.

**Qué queremos investigar.** Descriptor a descriptor, dos cosas separadas (si delata que algo va
mal y si dice cuál de las dos averías es) y, de paso, qué hace el cociente informado por la física
que el capítulo 4 construyó para delatar un caudalímetro que subestima.

**Qué datos utilizamos.** `data/processed/synthetic_v2_features.npy` y su `meta.parquet`, de
E5. **Solo la partición de entrenamiento**: 56 ventanas por condición de fallo y 112 sin fallo. Una
figura descriptiva no justifica abrir el conjunto reservado. La medida es el delta de Cliff que
`data/exploration.py` implementa, el mismo que usa E4.

In [ ]:
from mecanico_ia_exp.visualization.memoria import separabilidad_entre_mezclas_pobres

separabilidad = separabilidad_entre_mezclas_pobres()
orden = ["stft_bank1_mean", "ltft_bank1_max", "stft_bank1_max", "ltft_bank1_mean",
         "map_min", "ltft_bank1_delta_min", "ltft_bank1_n_events", "maf_min",
         "stft_bank1_slope", "maf_over_rpm_load"]
tabla_mezcla = separabilidad.set_index("descriptor").loc[orden, ["frente_a_sin_fallo", "entre_los_dos"]]
tabla_mezcla.columns = ["delata que algo va mal", "distingue una de la otra"]
print(tabla_mezcla.round(2).to_string())

In [ ]:
from mecanico_ia_exp.visualization.memoria import fig_mezcla_pobre

fig_mezcla_pobre()

**Qué muestra.** Los cuatro primeros descriptores (niveles y máximos de las
dos correcciones de combustible) delatan la avería con delta de Cliff entre 0,99 y 1,00;
distinguen una de otra entre 0,40 y 0,52. Los cuatro siguientes invierten el patrón: el mínimo de la
presión de colector delata 0,01 y distingue 0,80; el salto descendente del ajuste a largo plazo,
0,50 y 0,69; su recuento de eventos, 0,02 y 0,64; el mínimo del caudal, 0,13 y 0,64. El cociente
`maf_over_rpm_load` queda en 0,19 y 0,09.

**Qué podemos interpretar.** Lo que avisa y lo que distingue son conjuntos de descriptores distintos y casi
disjuntos. La lectura mecánica es coherente con el generador: la fuga de vacío entra por el colector
y eleva su presión, mientras la deriva del caudalímetro no toca esa señal; por eso la presión de
colector no sirve de aviso y sí de discriminante. Hay un resultado negativo: el cociente
informado por la física no cumple en este conjunto lo que su diseño esperaba. Su comentario lo
declaraba pendiente de comprobación sobre datos reales; sobre estos datos sintéticos no funciona.

**Qué no podemos afirmar.** Nada inferencial: el delta de Cliff es un tamaño de efecto **descriptivo** y las ventanas de una
misma sesión no son independientes (cada condición aporta 56 ventanas de 8 sesiones), de modo que no
se acompaña de contraste ni de intervalo. **Separabilidad descriptiva no es rendimiento de
clasificación demostrado** (apartado 10). Nada de esto se afirma de un vehículo real.

**Relación con la pregunta de investigación.** Sostiene PI.1 por su lado restrictivo: una explicación
que justificara la deriva del caudalímetro “porque los ajustes de combustible están altos” sería
indistinguible de la que justificaría una fuga de vacío. Por eso las plantillas de afirmación del
verificador no permiten formularla así. En la memoria es la **Figura 5.5** y la **Tabla B.5**.

## 12. Correlaciones: la estructura que un modelo puede explotar

**Qué queremos investigar.** Qué señales se mueven juntas, porque eso determina cuánta información
independiente hay realmente en las doce. Un par con correlación cercana a 1 indicaría redundancia.

**Qué datos utilizamos.** Las sesiones **sin fallo**, para que la estructura sea la del motor y no
la de las inyecciones.

In [ ]:
import numpy as np

from mecanico_ia_exp.visualization.style import DIVERGING

sanas = datos[datos.fault_id == "F00_normal"]
correlacion = sanas[list(SIGNAL_NAMES)].corr()

fig, eje = plt.subplots(figsize=(7.4, 6.2))
imagen = eje.imshow(correlacion, cmap=DIVERGING, vmin=-1, vmax=1)
eje.set_xticks(range(len(SIGNAL_NAMES)), [NOMBRE_SENAL[s] for s in SIGNAL_NAMES],
               rotation=55, ha="right", fontsize=7)
eje.set_yticks(range(len(SIGNAL_NAMES)), [NOMBRE_SENAL[s] for s in SIGNAL_NAMES], fontsize=7)
eje.grid(False)
for i in range(len(SIGNAL_NAMES)):
    for j in range(len(SIGNAL_NAMES)):
        valor = correlacion.iloc[i, j]
        if abs(valor) >= 0.55 and i != j:
            eje.text(j, i, f"{valor:.2f}", ha="center", va="center",
                     fontsize=6, color="#0b0b0b")
barra = fig.colorbar(imagen, ax=eje, shrink=0.8)
barra.set_label("coeficiente de Pearson (adimensional)", fontsize=7)
fig.suptitle(
    "Correlación entre señales en sesiones sin fallo",
    fontsize=10.5, fontweight="bold", x=0.0, ha="left",
)
fig.supxlabel(
    "E4 exploratorio (recalculado) · synthetic_sample, 3 sesiones sanas · "
    "se anotan los pares con |r| ≥ 0,55",
    fontsize=6.5, x=0.0, ha="left",
)
fig.tight_layout()
soporte.exportar(fig, "FIG-NB01-06_correlaciones", subdirectorio="01_exploracion_obd")
plt.show()

altas = (
    correlacion.where(~np.eye(len(correlacion), dtype=bool))
    .stack()
    .sort_values(ascending=False)
)
print("pares más correlacionados:")
print(altas[altas > 0.9].drop_duplicates().to_string())

**Qué muestra.** Dos grupos casi redundantes: carga, mariposa y presión de colector, con
correlaciones entre 0,992 y 0,997; régimen con velocidad, con 0,963. El caudal acompaña a los dos
grupos sin ser redundante con ninguno (0,85 con la carga, 0,75 con el régimen). El ajuste a corto
plazo, la sonda lambda y la temperatura de refrigerante apenas correlacionan con nada.

**Qué podemos interpretar.** Las doce señales **no aportan doce dimensiones de información**:
cinco de ellas se reducen, en la práctica, a dos. En el generador, la mariposa y la presión son
funciones afines de la carga más ruido. Eso tiene una consecuencia metodológica: buena parte de los
139 descriptores de ventana son redundantes entre sí. Una atribución que reparta importancia entre
señales acopladas la reparte de forma inestable.

**Qué no podemos afirmar.** Que esta estructura de correlaciones sea la de un vehículo real. Es la
que el generador construye, por diseño. Tampoco que la redundancia perjudique al bosque aleatorio,
que la tolera bien, ni que la correlación implique que una señal cause la otra: aquí las dos
dependen de la misma variable latente, la demanda del conductor.

## 13. Por qué la ventana dura 120 segundos

E5 no eligió el tamaño de ventana por costumbre. Lo midió: para cada duración candidata, cuántas
ventanas salen por sesión y cuánta separabilidad hay entre condiciones. La tabla está versionada.

In [ ]:
estudio = soporte.tabla("E5_estudio_ventana")
estudio

In [ ]:
from mecanico_ia_exp.visualization.memoria.base import CONDICION

fig, (izq, der) = plt.subplots(1, 2, figsize=(9.6, 3.6))

# Izquierda: el tamaño del efecto de cada condición frente a la duración.
for color, (fid, grupo) in zip(CATEGORICAL, estudio.groupby("fault_id"), strict=False):
    orden = grupo.sort_values("length_s")
    izq.plot(orden.length_s, orden.best_delta, marker="o", markersize=3.5,
             color=color, label=f"{fid.split('_', 1)[0]} {CONDICION[fid]}", linewidth=1.1)
suelo = float(estudio.noise_floor_p90.iloc[0])
izq.axhline(suelo, color="#898781", linestyle=":", linewidth=0.9)
izq.text(12, suelo + 0.02, f"suelo de ruido (p90) = {suelo:.2f}", fontsize=6.5, color="#52514e")
izq.axvline(120, color="#898781", linestyle="--", linewidth=0.9)
izq.set_xlabel("duración de la ventana (s)")
izq.set_ylabel("delta de Cliff del mejor descriptor\n(adimensional, 0–1)")
izq.set_title("Separabilidad por condición", fontsize=8.5)
izq.legend(fontsize=6, ncols=1, loc="center left", bbox_to_anchor=(0.0, 0.40),
           frameon=True, framealpha=0.9, facecolor="white", edgecolor="none")

# Derecha: cuántas ventanas quedan. Es la otra mitad del compromiso.
ventanas = estudio.drop_duplicates("length_s").sort_values("length_s")
der.plot(ventanas.length_s, ventanas.n_windows, marker="o", markersize=3.5,
         color=CATEGORICAL[0], linewidth=1.1)
der.axvline(120, color="#898781", linestyle="--", linewidth=0.9)
der.set_xlabel("duración de la ventana (s)")
der.set_ylabel("ventanas obtenidas (recuento)")
der.set_title("Ventanas disponibles", fontsize=8.5)
der.set_yscale("log")

fig.suptitle(
    "Por qué 120 s: más separabilidad cuesta menos ventanas",
    fontsize=10.5, fontweight="bold", x=0.0, ha="left",
)
fig.supxlabel(
    "E5 preprocesamiento · synthetic_v2 · fuente: artifacts/tables/E5_estudio_ventana.csv "
    "(cifras publicadas, no recalculadas aquí) · línea discontinua: duración elegida",
    fontsize=6.5, x=0.0, ha="left",
)
fig.tight_layout()
soporte.exportar(fig, "FIG-NB01-07_estudio_ventana", subdirectorio="01_exploracion_obd")
plt.show()

**Qué muestra.** A la izquierda, el tamaño del efecto del mejor descriptor de cada condición frente
a la duración de la ventana, con el suelo de ruido medido como referencia. A la derecha, cuántas
ventanas quedan, en escala logarítmica porque el recuento cae un orden de magnitud.

**Qué podemos interpretar.** Los dos paneles son las dos mitades de un compromiso. Cuatro de las
cinco condiciones ya superan el suelo de ruido con ventanas de 10 s; la que decide es el fallo de
encendido. A 60 s ya lo supera también, pero por poco: 0,4325 frente a un suelo de 0,4261. A 120 s
el margen crece a 0,6426 frente a 0,5133, a cambio de bajar de 1 308 a 588 ventanas; cada ventana
perdida hace la evaluación más frágil. 120 s es la duración que E5 eligió con ese compromiso, no
la primera en que las cinco superan el suelo.

**Qué no podemos afirmar.** Que 120 s sea óptimo para todas. Para el fallo de encendido, que es
episódico, una ventana larga **diluye** la firma: los pocos segundos alterados se promedian con los
que no lo están. La elección es un compromiso global y perjudica a esa condición en concreto.
Tampoco que el suelo de ruido sea un umbral de detectabilidad: es el percentil 90 del efecto medido
entre pares de sesiones **sanas**, es decir cuánto separa el azar. La figura dibuja un solo
suelo, el de 10 s (0,33), para todas las duraciones, cuando la tabla da uno por duración que sube
hasta 0,51 a 120 s: las comparaciones con el suelo deben hacerse con la tabla, no con la línea.

## 14. De la señal al descriptor: lo que el resumen conserva y lo que disuelve

El apartado anterior fija la ventana en 120 segundos. El modelo, sin
embargo, no ve la ventana: ve 139 números calculados sobre ella. Entre una cosa y otra hay una
traducción que nadie ha mirado todavía y es donde se pierde la información de los fallos
episódicos.

**Qué queremos investigar.** Seguir una misma ventana de fallo de encendido desde la señal a un
hertzio hasta los descriptores que de ella salen y comparar esos descriptores con los de las
ventanas sin fallo **del mismo régimen de conducción**. Compararlos con todas mediría el régimen y
no el fallo.

**Qué datos utilizamos.** Los dos primeros paneles, del análisis contrafactual T2
(`artifacts/contrafactual/`, descrito en el notebook 06, apartado 7.1); el tercero, de `synthetic_v2_features.npy`, que es el fichero que el
modelo consume. La ventana es de la partición de **entrenamiento**.

In [ ]:
from mecanico_ia_exp.visualization.memoria import fig_ventana_y_descriptor

fig_ventana_y_descriptor()

**Qué muestra.** En la señal, el fallo son tres muestras de un episodio
sobre ciento veinte. En el resumen, esa caída queda dentro de la banda de una desviación típica. En
los descriptores, la media se aparta 1,9 escalas robustas de la mediana sin fallo del mismo régimen
(687,9 frente a 690,3 min⁻¹), mientras el mínimo se aparta 9,6 (600,0 frente a 659,3), la desviación
típica 6,2 y el salto descendente máximo 4,3; el recuento de eventos pasa de 0 a 2.

**Qué podemos interpretar.** El descriptor que un lector miraría primero, la media, es el que
menos registra este fallo. La detección depende por completo de los descriptores que guardan el
extremo y la variación, que es la justificación empírica de la decisión del capítulo 4 de no limitar
el vector a momentos de primer orden.

**Qué no podemos afirmar.** Nada general: es **una ventana elegida como ilustración**. Procede de una sesión
en ralentí; ahí el efecto del fallo sobre el régimen queda **truncado por el límite físico
inferior** que el generador impone: el mínimo de 600,0 es el suelo del instrumento y no la
profundidad real de la caída. El apartamiento en escalas robustas dice cuán inusual es esta ventana,
no una probabilidad ni un contraste.

**Relación con la pregunta de investigación.** Delimita de dónde puede venir una explicación legítima
bajo PI.1: una afirmación sobre el fallo de encendido apoyada en la media del régimen no tendría
soporte observacional; la que se apoya en el mínimo y en el recuento de eventos sí lo tiene. En la
memoria es la **Figura 5.11**.

## 15. El reparto sin fuga

La unidad de reparto es la sesión (apartado 3). Es lo que separa una evaluación de un examen con
las respuestas.

In [ ]:
reparto_publicado = soporte.tabla("E5_reparto")
reparto_publicado

**Qué muestra la tabla.** Cuántas sesiones y ventanas van a cada partición, por condición, en el
conjunto oficial.

**Qué podemos interpretar.** Ninguna sesión aparece en dos particiones y cada condición está
presente en las tres. Dos ventanas consecutivas de una misma sesión comparten la mitad de sus
muestras (el paso es de 60 s sobre ventanas de 120 s), de modo que repartir por ventana habría
filtrado información de entrenamiento a evaluación de forma invisible para cualquier métrica.

**Qué no podemos afirmar.** Que el reparto por sesión elimine toda dependencia (apartado 16). La
evaluación mide generalización **a sesiones nuevas
del mismo generador**, no a vehículos nuevos.

La figura añade a los recuentos la **proporción entre clases**, que fija la referencia mínima
contra la que se compara cualquier modelo. Va apilada por partición para que se vea a la vez el
tamaño de cada clase y cómo se reparte.

In [ ]:
from mecanico_ia_exp.visualization.memoria import fig_particiones

fig = fig_particiones()
fig.suptitle(
    "Ventanas de 120 s por condición y partición en synthetic_v2 (reparto por sesión)",
    fontsize=9.5, fontweight="bold", x=0.01, y=1.0, ha="left", va="bottom",
)
soporte.exportar(fig, "FIG-NB01-08_particiones", subdirectorio="01_exploracion_obd")
plt.show()

**Qué muestra la figura.** La clase sin fallo reúne 112 ventanas de entrenamiento, 28 de validación
y 28 de prueba; cada condición de fallo, 56, 14 y 14. La clase mayoritaria **dobla** a cada una de
las cinco condiciones.

**Qué podemos interpretar.** El desequilibrio no es un defecto del muestreo: las sesiones averiadas
aportan sus ventanas **anteriores a la entrada del fallo**; esas ventanas son ventanas sin fallo.
De ahí que la clase mayoritaria alcance el 28,6 % de la validación, la exactitud de la referencia
mínima con que se compara el modelo de diagnóstico.

**Qué no podemos afirmar.** Nada sobre la dificultad de cada clase: dos condiciones con el mismo
número de ventanas pueden ser muy desiguales en lo que de ellas se puede aprender.

**Relación con la pregunta de investigación.** Fija la unidad de observación (la ventana, con la sesión como unidad
de reparto) sobre la que se cuentan después todas las decisiones del agente.

## 16. Resultados negativos y limitaciones

### 16.1 Resultados negativos

1. **El fallo de encendido apenas altera la señal.** Con episodios de 1 a 3 s y una tasa de
   aparición del 2 % por segundo, una fracción muy pequeña de las muestras etiquetadas difiere de su
   contrafactual. La etiqueta cubre la sesión; el fenómeno, no.
2. **El termostato no es observable en calentamiento** hasta que el motor alcanza su techo.
3. **Cinco de las doce señales forman dos grupos casi redundantes**: carga, mariposa y presión de
   colector (|r| ≥ 0,99) y régimen con velocidad (0,96). La dimensionalidad efectiva es menor que
   la nominal.
4. **La muestra versionada no cubre el régimen de calentamiento**; el apartado 8 lo ilustra.
5. **Las cifras de este notebook no son las publicadas de E4** (modalidad, apartado 4).

### 16.2 Limitaciones del diseño, que no se resuelven mirando más datos

- El conjunto es sintético. La validez externa no está establecida y no se afirma.
- El generador es paramétrico y no está calibrado contra datos reales; sus constantes sí contra el
  manual de taller, que es una cosa distinta.
- Las sesiones de una condición no son independientes entre sí: comparten generador,
  parametrización y plan. El reparto por sesión las hace **disjuntas**, no independientes en sentido
  estadístico.

## 17. Discusión

Lo que esta exploración aporta a la pregunta de investigación es una respuesta parcial y una
advertencia.

**La respuesta parcial.** Hay estructura y ninguna condición se separa trivialmente: un modelo de
diagnóstico tiene algo que aprender. Eso es la precondición de PI.1.

**La advertencia.** La relación entre **etiqueta** y **señal** no es uno a uno.
Hay ventanas etiquetadas con un fallo que no contienen información del fallo. Un modelo que falle en
ellas responde correctamente a lo que ve. Una métrica agregada que
las cuente como errores está midiendo, en parte, el límite de lo que el dato contiene y no la
calidad del modelo. Es una propiedad del generador, no una afirmación sobre vehículos reales.

La capacidad de **abstenerse** (PI.3) tiene
aquí su justificación empírica: existen situaciones en que la evidencia es insuficiente
por cómo se manifiesta el fallo, no por un defecto del sistema. Un sistema que explicara con confianza en
esas ventanas estaría inventando.

## 18. Conclusiones de este notebook

**Qué aprendimos.** Que las doce señales tienen la estructura que el diseño del generador declara;
que las cinco condiciones producen **cinco firmas distintas** en su señal principal (tres formas
temporales medidas como desviación), lo que hace que la elección de representación no sea neutral; y que al menos dos condiciones tienen ventanas
etiquetadas sin señal observable del fallo.

**Qué hipótesis recibió apoyo.** Que las condiciones desplazan las señales declaradas sin que
ninguna señal las separe por sí sola con un umbral.

**Qué no recibió apoyo.** Nada se refuta aquí, pero queda matizado que el desplazamiento sea
observable **durante todo el tramo en que el fallo está activo**: en dos de las cinco condiciones,
no lo es.

**Qué permanece sin demostrar.** Si estas firmas aparecen igual en telemetría real. Si la forma
temporal de la firma hace que una representación secuencial sea preferible: eso cambia dos cosas a
la vez y se trata en `02_comparacion_modelos.ipynb`.

**Cómo contribuye al TFM.** Es la base empírica del capítulo de datos y la justificación del tamaño
de ventana y del reparto por sesión.

## 19. Reproducibilidad

| | |
|---|---|
| **Python** | 3.11 |
| **Dependencias** | `uv sync --extra nb` |
| **Datos** | `data/samples/synthetic_sample.parquet` (versionado) y dos sesiones generadas en el notebook |
| **Semilla** | 20260901 para las sesiones ilustrativas |
| **Configuración** | ventana 120 s, paso 60 s (la de E5) |
| **Commit** | el que imprime la celda del apartado 4 |
| **Tiempo** | menos de dos minutos |
| **Memoria** | por debajo de 1 GB |
| **Recursos** | CPU; no hace falta acelerador |
| **Artefactos que produce** | ocho figuras en `notebooks/salida/01_exploracion_obd/`, en PNG y SVG |

**Qué se reproduce y qué no.** Las sesiones ilustrativas de calentamiento se regeneran idénticas:
el generador es determinista por semilla. Las cifras de E4 y E5 **no se reproducen**, se leen de sus
tablas versionadas.

## 20. Referencias y trazabilidad

**Dónde aparece esto en la memoria.** El reparto del conjunto oficial, `synthetic_v2`, es la
**Figura 5.1** y la separabilidad por señal, las **Figuras 5.2** y **5.3** (una por vía, nivel y
dispersión), en el análisis descriptivo del capítulo 5. Las doce señales y las cinco condiciones
están tabuladas en el anexo A, **Tablas A.2** y **A.3**. La **Figura A.2** es la versión publicada
de E4 sobre `synthetic_v1`, la primera figura del apartado 10.

- `docs/experiments/E2_datos_sinteticos.md`, `E4_eda_temporal.md`, `E5_preprocesamiento.md`
- `artifacts/tables/E4_descriptivos.csv`, `E4_separabilidad.csv`, `E5_estudio_ventana.csv`, `E5_reparto.csv`
- `artifacts/figures/FIG-E4-01_series_normal_vs_fallo.png`, `FIG-E4-02_separabilidad.png`
- `configs/calibracion_mazda3_bk.yaml`: cada constante del generador con su cita
- `src/mecanico_ia_exp/obd/`: generador, catálogo de fallos, auditoría de doce comprobaciones
- `docs/methodology/expresion-efectiva-del-fallo.md`: la definición de alteración observable
- `docs/diseno-pi1/08-analisis-por-fallo.md`: el cruce pendiente entre errores y expresión